In [0]:
dbutils.library.restartPython()

In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from pyspark.sql import functions as F
from lh import common, silver

cfg = common.load_config("dev")
spec = {"key": ["appt_id"], "allowed": {"status": ["completed", "cancelled"]},
        "columns": {"appt_id": "string", "phone": "phone", "appt_date": "date", "status": "code", "fee": "decimal"}}
rows = [("A1", "+91-7937946724", "2026-10-01", "Completed", "500", "f1.csv"),   # good
        (None, "7937946724", "2026-10-01", "completed", "500", "f1.csv"),       # missing_key
        ("A3", "7937946724", "not a date", "completed", "500", "f1.csv"),       # unreadable_value
        ("A4", "7937946724", "2026-10-01", "postponed", "500", "f1.csv"),       # value_not_allowed
        ("A5", "12345", "2026-10-01", "cancelled", "₹700", "f1.csv")]           # kept, phone warning
sample = spark.createDataFrame(rows, "appt_id string, phone string, appt_date string, status string, fee string, _source_file string")
got = {r["appt_id"]: (r["_rule"], r["_detail"], list(r["_soft"]))
       for r in silver.check(silver.apply(sample, spec), spec).collect()}
want = {"A1": (None, None, []), None: ("missing_key", "appt_id", []),
        "A3": ("unreadable_value", "appt_date", []), "A4": ("value_not_allowed", "status=postponed", []),
        "A5": (None, None, ["phone"])}
print("PASS" if got == want else f"FAIL {got}")

In [0]:
silver.ensure_tables(spark, cfg)
q, dq = f"{cfg['catalog']}.silver.quarantine", f"{cfg['catalog']}.ops.dq_results"
start = time.time()
counts = []
for run in (1, 2):                                   # the second run must not add rows
    good, failed = silver.split_and_log(spark, cfg, "_test", silver.apply(sample, spec), spec)
    counts.append((good.count(), spark.table(q).filter("table_name = '_test'").count()))
times = spark.table(q).filter("table_name = '_test'").selectExpr("min(first_seen) < max(last_seen) AS moved").first()["moved"]
display(spark.table(q).filter("table_name = '_test'").select("row_key", "rule", "detail", "status"))
spark.sql(f"DELETE FROM {q} WHERE table_name = '_test'")
spark.sql(f"DELETE FROM {dq} WHERE table_name = '_test'")
print(failed, counts, f"in {time.time() - start:.0f}s")
print("PASS" if counts == [(2, 3), (2, 3)] and times else "FAIL")

In [0]:
start = time.time()
total = 0
for table, s in silver.load_silver().items():
    checked = silver.check(silver.apply(spark.table(f"{cfg['catalog']}.bronze.{table}"), s), s)
    r = checked.agg(F.count("*").alias("n"), F.count("_rule").alias("bad"),
                    F.sum(F.when(F.size("_soft") > 0, 1).otherwise(0)).alias("warn")).first()
    total += r["bad"]
    print(f"{table:18} rows {r['n']:>5}  quarantine {r['bad']:>3}  warnings {r['warn']:>3}")
print(f"done in {time.time() - start:.0f}s", "PASS" if total == 0 else "CHECK")